In [28]:
import pandas as pd

In [29]:
# machine data
liw_feeders_1 = pd.read_csv("MSOM data external sharing\Machine data\LiW Feeders 1.csv")
liw_feeders_2 = pd.read_csv("MSOM data external sharing\Machine data\LiW Feeders 2.csv")
blenders = pd.read_csv("MSOM data external sharing\Machine data\Blenders.csv")
pressor = pd.read_csv("MSOM data external sharing\Machine data\Tablet Press.csv")
temperature = pd.read_csv("MSOM data external sharing\Machine data\Temperature.csv")
humidity = pd.read_csv("MSOM data external sharing\Machine data\Humidity.csv")

C:\Users\User\AppData\Local\Temp\ipykernel_19944\4085920741.py:2: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27) have mixed types. Specify dtype option on import or set low_memory=False.
  liw_feeders_1 = pd.read_csv("MSOM data external sharing\Machine data\LiW Feeders 1.csv")
C:\Users\User\AppData\Local\Temp\ipykernel_19944\4085920741.py:3: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  liw_feeders_2 = pd.read_csv("MSOM data external sharing\Machine data\LiW Feeders 2.csv")
C:\Users\User\AppData\Local\Temp\ipykernel_19944\4085920741.py:4: DtypeWarning: Columns (1,2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  blenders = pd.read_csv("MSOM data external sharing\Machine data\Blenders.csv")
C:\Users\User\AppData\Local\Temp\ipykernel_19944\4085920741.py:5: DtypeWarning: Columns (1

In [30]:
# concat liw_feeder1&2
liw_feeder = pd.concat([liw_feeders_1, liw_feeders_2], axis=1)
liw_feeder = liw_feeder.loc[:, ~liw_feeder.columns.str.contains("Unnamed:")]

In [31]:
#日期處理
def parse_mixed_date(date_str):
    try:
        # 嘗試美式日期
        return pd.to_datetime(date_str, format="%m/%d/%Y %H:%M")
    except ValueError:
        try:
            # 如果失敗，改用歐洲格式
            return pd.to_datetime(date_str, format="%d/%m/%Y %H:%M")
        except:
            return pd.NaT
        

liw_feeder["TimeStamp"] = liw_feeder["TimeStamp"].apply(parse_mixed_date)
blenders["TimeStamp"] = blenders["TimeStamp"].apply(parse_mixed_date)
pressor["TimeStamp"] = pressor["TimeStamp"].apply(parse_mixed_date)


liw_feeder = liw_feeder.dropna(subset=["TimeStamp"], how="any")
blenders = blenders.dropna(subset=['TimeStamp'], how="any")
pressor = pressor.dropna(subset=['TimeStamp'], how="any")

In [85]:
# contextual quality
logbook = pd.read_excel("MSOM data external sharing\Contextual quality data\Logbook Long Run Days.xlsx")
content = pd.read_excel("MSOM data external sharing\Contextual quality data\RM Content Uniformity.xlsx")
material_property = pd.read_excel("MSOM data external sharing\Contextual quality data\RM Material Properties.xlsx")
tablet_property = pd.read_excel("MSOM data external sharing\Contextual quality data\RM Tablet Properties and Drum Change.xlsx", sheet_name="Tablet properties", header=1)
drum_change = pd.read_excel("MSOM data external sharing\Contextual quality data\RM Tablet Properties and Drum Change.xlsx", sheet_name="Raw Material Drum change", header=1)

In [33]:
# tablet_property 處理
tablet_property = tablet_property.dropna(axis=0, how="all")

In [86]:
# drum_change 處理
api = drum_change.iloc[1:, 0:4].dropna(axis=0, how="all")
mgst = drum_change.iloc[1:, 4:8].dropna(axis=0, how="all")
lactose = drum_change.iloc[1:, 8:12].dropna(axis=0, how="all")
Ac_Di_Sol = drum_change.iloc[1:, 12:16].dropna(axis=0, how="all")
Avicel_102_PD1 = drum_change.iloc[1:, 16:20].dropna(axis=0, how="all")
Avicel_102_PD4 = drum_change.iloc[1:, 20:24].dropna(axis=0, how="all")

api = api[api["Est Refill time"] != "missing"]
mgst = mgst.rename(columns={"Date/Time.1": "Date/Time"})
lactose = lactose.rename(columns={"Date/Time.2": "Date/Time"})
Ac_Di_Sol = Ac_Di_Sol.rename(columns={"Date/Time.3": "Date/Time"})
Avicel_102_PD1 = Avicel_102_PD1.rename(columns={"Date/Time.4": "Date/Time"})
Avicel_102_PD4 = Avicel_102_PD4.rename(columns={"Date/Time.5": "Date/Time"})

In [87]:
# first day
# machine data
time_till_first_day = 60 * 60 * 15 + 60 * 17 - 5
liw_feeder = liw_feeder[:time_till_first_day]
blenders = blenders[:time_till_first_day]
pressor = pressor[:time_till_first_day]

# contextual data
api_first_day = api[:7]
mgst_first_day = mgst[:6]
lactose_first_day = lactose[:3]
Ac_Di_Sol_first_day = Ac_Di_Sol[:2]
Avicel_102_PD1_first_day = Avicel_102_PD1[:4]
Avicel_102_PD4_first_day = Avicel_102_PD4[:4]

In [88]:
# Based on the sampling interval of output type, find the data points of liw_feeder、blenders、pressor in each time interval.

def time(i: int, output_type: pd.DataFrame):
    if i == -1: 
        return f'Before {output_type.iloc[i+1]["Date/Time"]}'
    
    if i == len(output_type) - 1: 
        return f'After {output_type.iloc[i-1]["Date/Time"]}'
    
    return f'{output_type.iloc[i]["Date/Time"]} to {output_type.iloc[i+1]["Date/Time"]}'

def type_data_within_time(i: int, input_type: pd.DataFrame, output_type: pd.DataFrame):
    if i == -1:
        return input_type[input_type["TimeStamp"] <= output_type.iloc[i+1]["Date/Time"]]
    
    if i == len(output_type) - 1:
        return input_type[input_type["TimeStamp"] >= output_type.iloc[i-1]["Date/Time"]]
    
    return input_type[(input_type["TimeStamp"] > output_type.iloc[i]["Date/Time"]) & (input_type["TimeStamp"] < output_type.iloc[i+1]["Date/Time"])]

def data_divide(i: int, df: pd.DataFrame, input_key: str, output_key: str, input_type: pd.DataFrame, output_type:pd.DataFrame):
    df[input_key][output_key][time(i, output_type)] = type_data_within_time(i, input_type, output_type)

In [89]:
input_types = [liw_feeder, blenders, pressor]
input_keys = ["liw_feeder", "blenders", "pressor"]

output_types = [api_first_day, mgst_first_day, lactose_first_day, Ac_Di_Sol_first_day, Avicel_102_PD1_first_day, Avicel_102_PD4_first_day]
output_keys = ["API", "MGST", "Lactose", "Ac_Di_Sol", "Avicel_102_PD1", "Avicel_102_PD4"]

time_set = {}

for key in input_keys:
    time_set[key] = {}

for input_key in input_keys:
    for output_key in output_keys:
        time_set[input_key][output_key] = {}


print(time_set)

{'liw_feeder': {'API': {}, 'MGST': {}, 'Lactose': {}, 'Ac_Di_Sol': {}, 'Avicel_102_PD1': {}, 'Avicel_102_PD4': {}}, 'blenders': {'API': {}, 'MGST': {}, 'Lactose': {}, 'Ac_Di_Sol': {}, 'Avicel_102_PD1': {}, 'Avicel_102_PD4': {}}, 'pressor': {'API': {}, 'MGST': {}, 'Lactose': {}, 'Ac_Di_Sol': {}, 'Avicel_102_PD1': {}, 'Avicel_102_PD4': {}}}


In [96]:
for i, input_type in enumerate(input_types):
    for j, output_type in enumerate(output_types):
        k = -1
        while k < len(output_types[j]) - 1:
            print(output_keys[j], k)
            data_divide(k, time_set, input_keys[i], output_keys[j], input_type, output_type)
            k+=1

API -1
API 0
API 1
API 2
API 3
API 4
API 5
MGST -1
MGST 0
MGST 1
MGST 2
MGST 3
MGST 4
Lactose -1
Lactose 0
Lactose 1
Ac_Di_Sol -1
Ac_Di_Sol 0
Avicel_102_PD1 -1
Avicel_102_PD1 0
Avicel_102_PD1 1
Avicel_102_PD1 2
Avicel_102_PD4 -1
Avicel_102_PD4 0
Avicel_102_PD4 1
Avicel_102_PD4 2
API -1
API 0
API 1
API 2
API 3
API 4
API 5
MGST -1
MGST 0
MGST 1
MGST 2
MGST 3
MGST 4
Lactose -1
Lactose 0
Lactose 1
Ac_Di_Sol -1
Ac_Di_Sol 0
Avicel_102_PD1 -1
Avicel_102_PD1 0
Avicel_102_PD1 1
Avicel_102_PD1 2
Avicel_102_PD4 -1
Avicel_102_PD4 0
Avicel_102_PD4 1
Avicel_102_PD4 2
API -1
API 0
API 1
API 2
API 3
API 4
API 5
MGST -1
MGST 0
MGST 1
MGST 2
MGST 3
MGST 4
Lactose -1
Lactose 0
Lactose 1
Ac_Di_Sol -1
Ac_Di_Sol 0
Avicel_102_PD1 -1
Avicel_102_PD1 0
Avicel_102_PD1 1
Avicel_102_PD1 2
Avicel_102_PD4 -1
Avicel_102_PD4 0
Avicel_102_PD4 1
Avicel_102_PD4 2


In [97]:
time_set

{'liw_feeder': {'API': {'Before 2018-01-12 07:25:00': Empty DataFrame
   Columns: [TimeStamp, Feed Factor PD1, Feed Factor PD2, Feed Factor PD3, Feed Factor PD4, Feed Factor PD5, Feed Factor PD7, Screw RPM PD1, Screw RPM PD2, Screw RPM PD3, Screw RPM PD4, Screw RPM PD5, Screw RPM PD7, VolMode PD1, VolMode PD2, VolMode PD3, VolMode PD4, VolMode PD5, VolMode PD7, Massflow PD 1, Massflow PD 2, Massflow PD 3, Massflow PD 4, Massflow PD 5, Massflow PD 7, %  PD1, %  PD2, %  PD3, %  PD4, %  PD5, %  PD7, Estimated weight IBC PS1, Estimated weight IBC PS2, Estimated weight IBC PS3, Estimated weight IBC PS4, Estimated weight IBC PS5, Estimated weight IBC PS7, RefAct PD1, RefAct PD2, RefAct PD3, RefAct PD4, RefAct PD5, RefAct PD7, Net Weight PD1, Net Weight PD2, Net Weight PD3, Net Weight PD4, Net Weight PD5, Net Weight PD7, Totalizer PD1, Totalizer PD2, Totalizer PD3, Totalizer PD4, Totalizer PD5, Totalizer PD7]
   Index: []
   
   [0 rows x 55 columns],
   '2018-01-12 07:25:00 to 2018-01-12 09: